In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import folium
import unicodedata
import streamlit as st
import re
import math

local = Path('.')
files = sorted(local.glob('base/*.csv'))
coordenadas = pd.read_csv('coordenadas.csv', sep=',')
data = pd.concat((pd.read_csv(file, sep=';', encoding='latin-1') for file in files), ignore_index=True)

data['media_valor_passagem'] = data['media_valor_total'] / data['quantidade_bilhetes']

### LETRAS MAIÚSCULAS E REMOÇÃO ACENTUAÇÃO

In [2]:
data['ponto_origem_viagem'] = data['ponto_origem_viagem'].str.upper()
data['ponto_destino_viagem'] = data['ponto_destino_viagem'].str.upper()
data['tipo_servico'] = data['tipo_servico'].str.upper()
data['tipo_gratuidade'] = data['tipo_gratuidade'].str.upper()

### NORMATIZAÇÃO - RETIRAR ASSENTOS

In [3]:
def retirarAssento(text):
    if pd.isna(text):
        return text
    nfd = unicodedata.normalize('NFD', text)
    # lu - letra maiuscula, Ll - letra minuscula, Zs - espaço, Mn - assento/marca, Po - pontuação
    return ''.join(char for char in nfd if unicodedata.category(char) != 'Mn') 

colunas = ['ponto_origem_viagem', 'ponto_destino_viagem', 'tipo_servico', 'tipo_gratuidade']

for coluna in colunas:
    if coluna in data.columns:
        data[coluna] = data[coluna].apply(retirarAssento)

### SOMENTE SP

In [4]:
(data['ponto_origem_viagem'] == 'SAO PAULO/SP').value_counts()
porcentagem = ((data['ponto_origem_viagem'] == 'SAO PAULO/SP').value_counts([True])*100)
print(porcentagem)

dataSP = data[(data['ponto_origem_viagem'] == 'SAO PAULO/SP')]
display(dataSP.head(2))

ponto_origem_viagem
False    94.123199
True      5.876801
Name: proportion, dtype: float64


,mes_emissao_bilhete,mes_viagem,ponto_origem_viagem,ponto_destino_viagem,tipo_servico,tipo_gratuidade,media_valor_total,dp_valor_total,quantidade_bilhetes,media_valor_passagem
0,01/2025,01/2024,SAO PAULO/SP,EXTREMA/MG,EXECUTIVO,TARIFA NORMAL - SEM DESCONTO,48.0,0.0,3,16.0
80908,01/2025,01/2025,SAO PAULO/SP,ACAILANDIA/MA,SEMILEITO,AUTORIZACAO DE VIAGEM - PASSE LIVRE - ART. 1º ...,0.0,0.0,2,0.0


### CIDADES FERROVIA

In [5]:
(data['ponto_origem_viagem'] == 'LIMEIRA/SP').value_counts()
dataRailway = data[data['ponto_origem_viagem']
                   .isin(['SANTOS/SP','SAO VICENTE/SP',
                          'PRAIA GRANDE/SP','MONGAGUA/SP',
                          'ITANHAEM/SP','PERUIBE/SP',
                          'ITARIRI/SP','PEDRO DE TOLEDO/SP',
                          'MIRACATU/SP','JUQUIA/SP',
                          'REGISTRO/SP','JACUPIRANGA/SP',
                          'CAJATI/SP','SAO PAULO/SP',
                          'CURITIBA/PR','CUBATAO/SP',
                          'SAO VICENTE/SP'])]
display(dataRailway['ponto_origem_viagem'].unique()) #9 dos 13
display(dataRailway['ponto_origem_viagem'].value_counts())

<ArrowStringArray>
[   'SAO PAULO/SP',      'CUBATAO/SP',     'CURITIBA/PR',     'ITANHAEM/SP',
      'PERUIBE/SP', 'PRAIA GRANDE/SP',     'REGISTRO/SP',       'SANTOS/SP',
  'SAO VICENTE/SP',     'MIRACATU/SP',       'CAJATI/SP',  'JACUPIRANGA/SP']
Length: 12, dtype: str

ponto_origem_viagem
SAO PAULO/SP       124521
CURITIBA/PR         33893
SANTOS/SP            8679
PERUIBE/SP            905
ITANHAEM/SP           864
SAO VICENTE/SP        703
REGISTRO/SP           456
PRAIA GRANDE/SP       453
CUBATAO/SP            439
MIRACATU/SP           123
CAJATI/SP             101
JACUPIRANGA/SP         91
Name: count, dtype: int64

In [6]:
# PASSAGENS
display(dataRailway.groupby('ponto_origem_viagem')['quantidade_bilhetes'].sum().sort_values())

ponto_origem_viagem
CUBATAO/SP            1822
MIRACATU/SP           1947
SAO VICENTE/SP        4528
PRAIA GRANDE/SP       4670
JACUPIRANGA/SP        4872
CAJATI/SP             5572
ITANHAEM/SP           6561
PERUIBE/SP            9479
REGISTRO/SP          10228
SANTOS/SP           140506
CURITIBA/PR        1448298
SAO PAULO/SP       5327318
Name: quantidade_bilhetes, dtype: int64

In [7]:
coordenadasMapa = {}
for _, row in coordenadas.iterrows():
    city = row['Cidade']
    print(city)
    coordenadasMapa[city] = [row['Latitude'], row['Longitude']]
grouped = dataRailway.groupby('ponto_origem_viagem')['quantidade_bilhetes'].sum().sort_values()

SAO PAULO/SP
CUBATAO/SP
SANTOS/SP
SAO VICENTE/SP
PRAIA GRANDE/SP
MONGAGUA/SP
ITANHAEM/SP
PERUIBE/SP
ITARIRI/SP
PEDRO DE TOLEDO/SP
MIRACATU/SP
JUQUIA/SP
REGISTRO/SP
JACUPIRANGA/SP
CAJATI/SP
CURITIBA/PR


### GERAR MAPA

In [19]:
import math
mapa = folium.Map(
    location=[-24.4971,-47.8449],
    zoom_start=9,
    tiles='OpenStreetMap'
)

# Dicionário com as coordenadas
coordenadasMapa = {}
for _, row in coordenadas.iterrows():
    city = row['Cidade']
    coordenadasMapa[city] = [row['Latitude'], row['Longitude']]

# Soma de passagens por cidade
grouped = dataRailway.groupby('ponto_origem_viagem')['quantidade_bilhetes'].sum().sort_values()

# Parâmetros de escala visual
raio_min = 4
fator_escala = 0.08
reducao_sao_paulo = 0.65

for cidade, totalPassagens in grouped.items():
    if cidade in coordenadasMapa:
        tamanho = raio_min + math.sqrt(totalPassagens) * fator_escala

        if cidade == 'SAO PAULO/SP':
            tamanho = tamanho * reducao_sao_paulo

        folium.CircleMarker(
            location=coordenadasMapa[cidade],
            radius=tamanho,
            tooltip=f"<b>{cidade}</b><br>Passagens: {totalPassagens}",
            color='#3186cc',
            fill=True,
            fillColor='#3186cc',
            fillOpacity=0.6,
            weight=2
        ).add_to(mapa)

# Monta a rota uma única vez
rota = [
    coordenadasMapa[row['Cidade']]
    for _, row in coordenadas.iterrows()
    if row['Cidade'] in coordenadasMapa
]

folium.PolyLine(
    locations=rota,
    color='orange',
    weight=5,
    opacity=0.9
).add_to(mapa)

mapa.save('Ferrovia_Santos_Cajati.html')

### GERAR MAPA

### LIMPEZA COLUNA GRATUIDADE

In [10]:
dataRailway['tipo_gratuidade'] = dataRailway['tipo_gratuidade'].str.split('-').str[0]
display(dataRailway['tipo_gratuidade'].unique())

array(['TARIFA NORMAL ', 'TARIFA PROMOCIONAL ', 'GRATUIDADE DE CRIANCA ',
       'BILHETE DE VIAGEM DO IDOSO 100% ',
       'BILHETE DE VIAGEM DO IDOSO 50% ',
       'GRATUIDADE JOVEM DE BAIXA RENDA 100% ', 'AUTORIZACAO DE VIAGEM ',
       'GRATUIDADE JOVEM DE BAIXA RENDA 50% ',
       'PASSE LIVRE AUDITORES E AGENTES DO TRABALHO ', nan], dtype=object)

### GERANDO EXCEL FERROVIA

In [11]:
dataRailway['media_valor_passagem'] = round(dataRailway['media_valor_passagem'],2)
dataRailway.to_excel('dataRailway.xlsx', index=False)

### GERANDO EXCEL MUNICÍPIOS ÚNICOS

In [12]:
dataUnique = pd.DataFrame(data['ponto_destino_viagem'].unique(), columns=['ponto_destino_viagem'])
dataUnique.to_excel('dataDestiny.xlsx')
display(dataUnique)

,ponto_destino_viagem
0,EXTREMA/MG
1,BRASILIA/DF
2,ARAGUAINA/TO
3,SAO PAULO/SP
4,ASSIS CHATEAUBRIAND/PR
...,...
2040,CAPIVARI DE BAIXO/SC
2041,SAO FRANCISCO DO MARANHAO/MA
2042,ALHANDRA/PB
2043,IGARAPE/MG
